## **Random Forest Cultivation**
### Experimenting with Feature Selection to Improve Performance
---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Libraries imported successfully!


In [ ]:
# Restore variables from ThimkersProcessing
%store -r

print("Variables restored successfully!")
print(f"\nDataset shapes:")
print(f"Training: {X_train_scaled.shape}")
print(f"Validation: {X_val_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")
print(f"\nBest Random Forest:")
print(f"Best hyperparameters: {best_rf_label}")
print(f"n_estimators: {best_n}")
print(f"max_depth: {best_depth}")
print(f"bootstrap: {best_bootstrap}")
print(f"Baseline accuracy: {acc_rf:.4f}")

Variables restored successfully!

Dataset shapes:
  Training:   (40818, 395)
  Validation: (9838, 395)
  Test:       (9839, 395)

Baseline Random Forest:
  Best hyperparameters: depth=None/bootstrap=True
  n_estimators: 500
  max_depth: None
  bootstrap: True
  Baseline accuracy: 0.7515


## **Experiment: Iterative Feature Elimination**
### Strategy: Remove least important features one by one and track performance
---

In [ ]:
# Get feature names and initial feature importances
feature_names = X_clean.columns.tolist()
n_features = len(feature_names)

# Get initial importances from the baseline model
initial_importances = rf_best.feature_importances_

print(f"Total features: {n_features}")

Total features: 395
Starting iterative feature elimination...
This will train 395 Random Forest models (one per feature removal)

Note: This may take several minutes depending on your dataset size.


In [5]:
# Iterative Feature Elimination Loop
# Start with all features, remove least important one at a time

# Convert scaled arrays back to dataframes for easier feature manipulation
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_val_df = pd.DataFrame(X_val_scaled, columns=feature_names)
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Track results
n_features_list = []
train_errors = []
val_errors = []
test_errors = []
removed_features = []

# Start with all features
current_features = feature_names.copy()

print("Starting elimination loop...\n")

for iteration in range(n_features):
    n_current = len(current_features)
    
    # Train Random Forest with current feature set
    rf = RandomForestClassifier(
        n_estimators=best_n,
        max_depth=best_depth,
        bootstrap=best_bootstrap,
        random_state=42,
        n_jobs=-1
    )
    
    # Get current feature subset
    X_train_current = X_train_df[current_features]
    X_val_current = X_val_df[current_features]
    X_test_current = X_test_df[current_features]
    
    # Fit and evaluate
    rf.fit(X_train_current, y_train)
    
    train_acc = rf.score(X_train_current, y_train)
    val_acc = rf.score(X_val_current, y_val)
    test_acc = rf.score(X_test_current, y_test)
    
    # Store results
    n_features_list.append(n_current)
    train_errors.append(1 - train_acc)
    val_errors.append(1 - val_acc)
    test_errors.append(1 - test_acc)
    
    # Print progress for every iteration
    print(f"Features: {n_current:>3} | Train Err: {1-train_acc:.4f} | Val Err: {1-val_acc:.4f} | Test Err: {1-test_acc:.4f}")
    
    # Find least important feature and remove it
    if n_current > 1:  # Keep at least 1 feature
        importances = rf.feature_importances_
        least_important_idx = np.argmin(importances)
        least_important_feature = current_features[least_important_idx]
        removed_features.append(least_important_feature)
        current_features.pop(least_important_idx)
    else:
        break

print("\n✓ Feature elimination complete!")

Starting elimination loop...

Features: 395 | Train Err: 0.0002 | Val Err: 0.2552 | Test Err: 0.2485
Features: 395 | Train Err: 0.0002 | Val Err: 0.2552 | Test Err: 0.2485
Features: 394 | Train Err: 0.0002 | Val Err: 0.2576 | Test Err: 0.2475
Features: 394 | Train Err: 0.0002 | Val Err: 0.2576 | Test Err: 0.2475
Features: 393 | Train Err: 0.0002 | Val Err: 0.2594 | Test Err: 0.2474
Features: 393 | Train Err: 0.0002 | Val Err: 0.2594 | Test Err: 0.2474
Features: 392 | Train Err: 0.0002 | Val Err: 0.2578 | Test Err: 0.2480
Features: 392 | Train Err: 0.0002 | Val Err: 0.2578 | Test Err: 0.2480
Features: 391 | Train Err: 0.0002 | Val Err: 0.2558 | Test Err: 0.2460
Features: 391 | Train Err: 0.0002 | Val Err: 0.2558 | Test Err: 0.2460
Features: 390 | Train Err: 0.0002 | Val Err: 0.2554 | Test Err: 0.2475
Features: 390 | Train Err: 0.0002 | Val Err: 0.2554 | Test Err: 0.2475
Features: 389 | Train Err: 0.0002 | Val Err: 0.2594 | Test Err: 0.2493
Features: 389 | Train Err: 0.0002 | Val Err: 0.

In [7]:
# Plot Error vs Number of Features
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=n_features_list, y=train_errors,
    mode='lines+markers', name='Train Error',
    line=dict(color='royalblue', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=n_features_list, y=val_errors,
    mode='lines+markers', name='Validation Error',
    line=dict(color='orange', width=2),
    marker=dict(size=4)
))

fig.add_trace(go.Scatter(
    x=n_features_list, y=test_errors,
    mode='lines+markers', name='Test Error',
    line=dict(color='green', width=2),
    marker=dict(size=4)
))

# Mark the best validation error point
best_val_idx = np.argmin(val_errors)
best_n_features = n_features_list[best_val_idx]
best_val_error = val_errors[best_val_idx]

fig.add_vline(
    x=best_n_features, 
    line_dash='dash', 
    line_color='red',
    annotation_text=f'Best: {best_n_features} features<br>Val Error: {best_val_error:.4f}',
    annotation_position='top right'
)

fig.update_layout(
    title='Random Forest - Error Rate vs Number of Features<br>(Iterative Least Important Feature Elimination)',
    xaxis_title='Number of Features',
    yaxis_title='Error Rate',
    template='plotly_white',
    height=600,
    hovermode='x unified'
)

fig.update_xaxes(autorange='reversed')  # Reverse so we go from many to few features

fig.show()

print(f"\n{'='*60}")
print(f"RESULTS SUMMARY")
print(f"{'='*60}")
print(f"Baseline (all {n_features} features):")
print(f"  Train Error: {train_errors[0]:.4f}")
print(f"  Val Error:   {val_errors[0]:.4f}")
print(f"  Test Error:  {test_errors[0]:.4f}")
print(f"\nBest Configuration ({best_n_features} features):")
print(f"  Train Error: {train_errors[best_val_idx]:.4f}")
print(f"  Val Error:   {val_errors[best_val_idx]:.4f} ⭐")
print(f"  Test Error:  {test_errors[best_val_idx]:.4f}")
print(f"\nImprovement:")
print(f"  Val Error Change:  {val_errors[0] - best_val_error:+.4f}")
print(f"  Test Error Change: {test_errors[0] - test_errors[best_val_idx]:+.4f}")
print(f"  Features Removed:  {n_features - best_n_features}")


RESULTS SUMMARY
Baseline (all 395 features):
  Train Error: 0.0002
  Val Error:   0.2552
  Test Error:  0.2485

Best Configuration (236 features):
  Train Error: 0.0002
  Val Error:   0.2495 ⭐
  Test Error:  0.2435

Improvement:
  Val Error Change:  +0.0057
  Test Error Change: +0.0050
  Features Removed:  159


## **Analysis: Top Removed Features**
### Let's see which features were eliminated first (least important)
---

In [8]:
# Show first 30 features that were removed (least important)
print("First 30 Features Removed (Least Important):")
print("="*60)
for i, feature in enumerate(removed_features[:30], 1):
    print(f"{i:>3}. {feature}")

# Show last 30 features remaining (most important)
n_remaining = min(30, len(removed_features))
print(f"\n\nLast {n_remaining} Features Removed (Most Important):")
print("="*60)
for i, feature in enumerate(removed_features[-n_remaining:], 1):
    print(f"{i:>3}. {feature}")

First 30 Features Removed (Least Important):
  1. ai_orchestration_martian
  2. aimodel_reka_flash_3_or_other_reka_models
  3. ai_orchestration_smol_agi
  4. ai_observe_metero
  5. ai_observe_opik
  6. ai_observe_helicone
  7. ai_orchestration_phidata
  8. ai_knowledge_letta
  9. ai_external_openhands_formerly_opendevin
 10. ai_knowledge_zep
 11. ai_orchestration_lyzr
 12. ai_orchestration_agno
 13. ai_observe_vectra_ai
 14. ai_observe_adversarial_robustness_toolbox_art
 15. ai_external_glean_enterprise_agents
 16. ai_observe_protect_ai
 17. lang_mojo
 18. ai_observe_arize
 19. aimodel_cohere:_command_a
 20. ai_knowledge_weaviate
 21. ai_orchestration_smolagents
 22. region_caribbean
 23. ai_observe_galileo
 24. ai_knowledge_lancedb
 25. region_central_asia
 26. ai_knowledge_mem0
 27. db_datomic
 28. ai_orchestration_haystack
 29. ai_knowledge_milvus
 30. region_middle_africa


Last 30 Features Removed (Most Important):
  1. db_mysql
  2. aiagentchange_yes_somewhat
  3. platform_docker